In [1]:
# ===== CELL 1: FILE UPLOAD ONLY =====
# Run this cell first

from google.colab import files
import io
import pandas as pd

print("Please upload 'nse_indexes.csv' and 'stocks_df.csv'")
uploaded = files.upload()

# Helper to find uploaded filenames
def find_file(keywords):
    for fname in uploaded.keys():
        for k in keywords:
            if k.lower() in fname.lower():
                return fname
    return None

index_file = find_file(["nse", "index"])
stock_file = find_file(["stock"])

if index_file is None or stock_file is None:
    raise FileNotFoundError("Could not detect required files. Please upload both CSVs.")

indexes = pd.read_csv(io.BytesIO(uploaded[index_file]))
stocks = pd.read_csv(io.BytesIO(uploaded[stock_file]))

print("Files loaded successfully")
print("Indexes shape:", indexes.shape)
print("Stocks shape:", stocks.shape)


Please upload 'nse_indexes.csv' and 'stocks_df.csv'


Saving nse_indexes.csv to nse_indexes.csv
Saving stocks_df.csv to stocks_df.csv
Files loaded successfully
Indexes shape: (87536, 8)
Stocks shape: (4094387, 8)


In [17]:
# ===== CELL 2: Absolute-Risk (Risk-Parity) pipeline =====
# Assumptions / settings (edit if needed)
RISK_FREE_RATE = 0.07            # For reporting only
BENCHMARK_NAME = "NIFTY 500"      # not used for constraints in absolute-risk mode
MAX_STOCK_WEIGHT = 0.15          # 10%
SECTOR_CAP = 0.30                # 20% (you can change)
BETA_LOW, BETA_HIGH = 0.5, 1.2   # Widened beta band for absolute-risk
ADV_POSITION_PCT = 0.07          # position <= 5% of ADV
ANNUAL_TURNOVER_CAP = 2.5       # 250% annual
REBALANCES_PER_YEAR = 12
TURNOVER_PER_REBALANCE = ANNUAL_TURNOVER_CAP / REBALANCES_PER_YEAR
MIN_DATA_RATIO = 0.75            # require at least 80% data presence per ticker

# Install packages
!pip install --quiet scikit-learn yfinance statsmodels

import numpy as np, pandas as pd, math, time, warnings
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import yfinance as yf
warnings.filterwarnings("ignore")
start_time = time.time()

# ---------- Basic checks & normalize ----------
indexes.columns = indexes.columns.str.strip()
stocks.columns = stocks.columns.str.strip()
indexes['Date'] = pd.to_datetime(indexes['Date'])
stocks['Date'] = pd.to_datetime(stocks['Date'])

# Standardize ticker column name
if 'Stock' not in stocks.columns:
    if 'Ticker' in stocks.columns:
        stocks.rename(columns={'Ticker':'Stock'}, inplace=True)
    elif 'symbol' in stocks.columns:
        stocks.rename(columns={'symbol':'Stock'}, inplace=True)
    else:
        raise ValueError("stocks must have a 'Stock' or 'Ticker' column")

stocks['Stock'] = stocks['Stock'].astype(str).str.upper().str.strip()

# ---------- Fetch sector mapping (Wikipedia then yfinance fallback) ----------
def fetch_nifty500_sector_map():
    mapping = {}
    try:
        tables = pd.read_html("https://en.wikipedia.org/wiki/NIFTY_500", flavor='bs4')
        for tbl in tables:
            cols = [str(c).lower() for c in tbl.columns]
            if any('symbol' in c or 'ticker' in c or 'code' in c for c in cols) and any('sector' in c or 'industry' in c or 'group' in c for c in cols):
                # heuristics
                tick_col = [c for c in tbl.columns if any(k in str(c).lower() for k in ('symbol','ticker','code'))][0]
                sec_col  = [c for c in tbl.columns if any(k in str(c).lower() for k in ('sector','industry','group'))][0]
                tmp = tbl[[tick_col, sec_col]].dropna()
                tmp[tick_col] = tmp[tick_col].astype(str).str.upper().str.strip()
                for _, r in tmp.iterrows():
                    t = str(r[tick_col]).strip()
                    s = str(r[sec_col]).strip()
                    # extract token if in parens
                    if "(" in t and ")" in t:
                        inside = t[t.find("(")+1:t.rfind(")")]
                        if len(inside)<=10:
                            mapping[inside.upper()] = s
                            continue
                    # simple tokens (single-word short)
                    if len(t.split())==1 and len(t)<=10:
                        mapping[t] = s
    except Exception:
        pass
    return mapping

sector_map = fetch_nifty500_sector_map()
print("Fetched sector_map size from Wikipedia:", len(sector_map))

# fallback with yfinance for tickers in file
if len(sector_map) < 200:
    uniq = sorted(set(stocks['Stock'].unique()))
    for t in uniq:
        if t in sector_map:
            continue
        yf_t = t if "." in t else f"{t}.NS"
        try:
            info = yf.Ticker(yf_t).info
            sect = info.get('sector') or info.get('industry')
            if sect:
                sector_map[t] = sect
        except Exception:
            continue
    print("After yfinance fallback, mapping size:", len(sector_map))

# ---------- Universe: strict NIFTY500 alignment if possible ----------
stocks_in_file = set(stocks['Stock'].unique())
nifty_parsed = set(sector_map.keys())
if len(nifty_parsed) > 0:
    universe = sorted(list(stocks_in_file.intersection(nifty_parsed)))
    if len(universe) == 0:
        universe = sorted(list(stocks_in_file))
else:
    universe = sorted(list(stocks_in_file))

print("Universe size after NIFTY500 strictness:", len(universe))

# assign sector, missing -> Others
sector_for_universe = {t: sector_map.get(t, "Others") for t in universe}

# ---------- Filter stocks to universe, merge benchmark (benchmark only for reporting) ----------
stocks = stocks[stocks['Stock'].isin(universe)].copy()
indexes_bench = indexes[indexes['Index'] == BENCHMARK_NAME][['Date','Close']].rename(columns={'Close':'Benchmark_Close'})
# remove existing Benchmark_Close to avoid duplicates
if 'Benchmark_Close' in stocks.columns:
    stocks = stocks.drop(columns=['Benchmark_Close'])
stocks = stocks.merge(indexes_bench, on='Date', how='left')

# ---------- Compute returns ----------
stocks = stocks.sort_values(['Stock','Date']).reset_index(drop=True)
stocks['Stock_Return'] = stocks.groupby('Stock')['Close'].pct_change()
stocks['Market_Return'] = stocks['Benchmark_Close'].pct_change()
stocks = stocks.dropna(subset=['Stock_Return']).copy()  # allow market_return NaN if benchmark missing

# ---------- Liquidity: ADV last 90 days ----------
if 'Volume' not in stocks.columns:
    raise ValueError("stocks must include 'Volume' to compute ADV")
adv_series = stocks.groupby('Stock').apply(lambda df: df.sort_values('Date').tail(90)['Volume'].mean()).rename('ADV')
avg_price = stocks.groupby('Stock').apply(lambda df: df.sort_values('Date').tail(90)['Close'].mean()).rename('AvgPrice')
adv_value = (adv_series * avg_price).fillna(0)

# ---------- Returns matrix ----------
returns = stocks.pivot_table(index='Date', columns='Stock', values='Stock_Return')
good_cols = returns.columns[returns.isna().mean() <= (1 - MIN_DATA_RATIO)]
returns = returns[good_cols].copy()
returns = returns.fillna(returns.mean())
tickers = list(returns.columns)
n = len(tickers)
print(f"Using {n} tickers for optimization (after data sufficiency filter).")

# ---------- Covariance (Ledoit-Wolf) + annual returns ----------
lw = LedoitWolf().fit(returns.values)
cov_matrix = lw.covariance_ * 252
ann_returns = returns.mean() * 252

# ---------- Market & beta ----------
market = stocks.groupby('Date')['Market_Return'].first().reindex(returns.index).fillna(0)
market_var = market.var() * 252 if len(market)>1 else 1.0
beta_vec = np.array([ (np.cov(returns[t].values, market.values)[0,1] * 252) / (market_var if market_var>0 else 1.0) for t in tickers ])

# ---------- Stage 1: unconstrained-ish risk parity warm start ----------
sector_map_final = {t: sector_for_universe.get(t,"Others") for t in tickers}
sectors = sorted(list(set(sector_map_final.values())))
init = np.zeros(n)
for s in sectors:
    members = [i for i,tk in enumerate(tickers) if sector_map_final[tk]==s]
    if len(members)==0: continue
    w_s = 1.0 / len(sectors)
    for i in members:
        init[i] = w_s / len(members)
if init.sum()==0: init = np.ones(n)/n
else: init = init / init.sum()

def risk_parity_obj(w, cov):
    rc = w * (cov.dot(w))
    t = rc.mean()
    return np.sum((rc - t)**2)

bnds_stage1 = tuple((0.0, MAX_STOCK_WEIGHT) for _ in range(n))
cons_stage1 = ({'type':'eq', 'fun': lambda w: np.sum(w)-1.0},)

print("Stage 1: running risk-parity warm-start.")
res1 = minimize(risk_parity_obj, init, args=(cov_matrix,), method='SLSQP', bounds=bnds_stage1, constraints=cons_stage1, options={'maxiter':2000, 'ftol':1e-12})
if not res1.success:
    print("Stage1 warning:", res1.message)
w_stage1 = np.maximum(res1.x, 0.0)
w_stage1 = w_stage1 / w_stage1.sum()

# ---------- Stage 2: projection to enforce constraints (NO tracking-error constraint in absolute-risk) ----------
print("Stage 2: projecting to constraints (sector caps, liquidity, beta band, turnover)")

# build bounds combining liquidity & MAX_STOCK_WEIGHT (heuristic cap based on median ADV)
median_adv = np.median(list(adv_value[adv_value>0]) or [1.0])
bounds = []
for t in tickers:
    liq_val = adv_value.get(t, 0.0)
    if liq_val > 0:
        liq_cap_pct = ADV_POSITION_PCT * (liq_val / median_adv)
        ub = min(MAX_STOCK_WEIGHT, max(1e-6, liq_cap_pct))
    else:
        ub = MAX_STOCK_WEIGHT
    bounds.append((0.0, ub))

# constraints
constraints = []
constraints.append({'type':'eq', 'fun': lambda w: np.sum(w) - 1.0})

# sector caps
for s in sectors:
    idxs = [i for i,t in enumerate(tickers) if sector_map_final[t]==s]
    if len(idxs)==0: continue
    constraints.append({'type':'ineq', 'fun': (lambda w, idxs=idxs, cap=SECTOR_CAP: cap - np.sum(w[idxs]))})

# beta band (wider in absolute-risk)
constraints.append({'type':'ineq', 'fun': lambda w: np.dot(w, beta_vec) - BETA_LOW})
constraints.append({'type':'ineq', 'fun': lambda w: BETA_HIGH - np.dot(w, beta_vec)})

# turnover constraint (assume prev equal-weight; replace prev_w with real holdings if available)
prev_w = np.ones(n) / n
constraints.append({'type':'ineq', 'fun': lambda w: TURNOVER_PER_REBALANCE - np.sum(np.abs(w - prev_w))})

# discourage fully equal weights (small dispersion)
constraints.append({'type':'ineq', 'fun': lambda w: np.std(w) - 0.005})

# projection objective: minimize distance to stage1 weights (can include slight return tilt if desired)
def projection_obj(w, target):
    # keep mostly close to risk-parity but allow return tilt via small secondary term if you want
    alpha = 0.95
    dist = np.sum((w - target)**2)
    ret_term = -np.dot(w, ann_returns.reindex(tickers).fillna(0).values)
    return alpha * dist + (1 - alpha) * ret_term

x0 = w_stage1.copy()
res2 = minimize(projection_obj, x0, args=(w_stage1,), method='SLSQP', bounds=bounds, constraints=constraints, options={'maxiter':3000, 'ftol':1e-12})

if not res2.success:
    print("Stage2 projection did NOT converge cleanly:", res2.message)
    # attempt one retry with looser tol
    res2 = minimize(projection_obj, res2.x if res2.x is not None else x0, args=(w_stage1,), method='SLSQP', bounds=bounds, constraints=constraints, options={'maxiter':5000, 'ftol':1e-9})
    if not res2.success:
        print("Stage2 retry failed:", res2.message)
        # sector-safe repair fallback (enforce sector caps manually then renormalize)
        w_tmp = np.clip(w_stage1, 0, None)
        for s in sectors:
            idxs = [i for i,t in enumerate(tickers) if sector_map_final[t]==s]
            if not idxs: continue
            sector_w = w_tmp[idxs].sum()
            if sector_w > SECTOR_CAP:
                scale = SECTOR_CAP / sector_w
                for i in idxs:
                    w_tmp[i] *= scale
        if w_tmp.sum() > 0:
            w_final = w_tmp / w_tmp.sum()
        else:
            w_final = np.ones(n) / n
    else:
        w_final = np.maximum(res2.x, 0.0)
else:
    w_final = np.maximum(res2.x, 0.0)

# ensure normalization
if w_final.sum() <= 0:
    w_final = np.ones(n) / n
else:
    w_final = w_final / w_final.sum()

# ---------- Post checks & outputs ----------
portfolio = pd.DataFrame({'Stock': tickers, 'Weight': w_final})
portfolio = portfolio[portfolio['Weight'] > 1e-6].sort_values('Weight', ascending=False).reset_index(drop=True)
portfolio['Sector'] = portfolio['Stock'].map(sector_map_final)

total_weight = portfolio['Weight'].sum()
num_positions = len(portfolio)
top10 = portfolio.head(26).copy()
sector_exposure = portfolio.groupby('Sector')['Weight'].sum().sort_values(ascending=False)

expected_return = np.dot(portfolio.set_index('Stock').reindex(tickers)['Weight'].fillna(0).values, ann_returns.reindex(tickers).fillna(0).values)
portfolio_vol = math.sqrt(max(np.dot(w_final, cov_matrix).dot(w_final), 0.0))
portfolio_beta = np.dot(w_final, beta_vec)

portfolio_daily = returns.dot(w_final)
tracking_error_emp = portfolio_daily.sub(market).std() * math.sqrt(252)
turnover = np.sum(np.abs(w_final - prev_w))

print("\n--- Absolute-Risk (Risk-Parity) Portfolio Summary ---")
print(f"Elapsed time: {time.time()-start_time:.1f}s")
print("Positions:", num_positions)
print("Total weight:", total_weight)
print("Top ", num_positions,":")
display(top10.style.format({"Weight":"{:.4f}"}))
print("\nSector exposures:")
display(sector_exposure.to_frame("Weight").style.format({"Weight":"{:.4f}"}))
print(f"\nExpected annual return (approx): {expected_return:.4f} ({expected_return*100:.2f}%)")
print(f"Annual volatility (approx): {portfolio_vol:.4f} ({portfolio_vol*100:.2f}%)")
print(f"Portfolio beta: {portfolio_beta:.4f}")
print(f"Tracking error vs benchmark (empirical): {tracking_error_emp:.4f} ({tracking_error_emp*100:.2f}%)")
print(f"Per-rebalance turnover (L1): {turnover:.4f} (cap: {TURNOVER_PER_REBALANCE:.4f})")

# save results
portfolio.to_csv("absolute_risk_parity_portfolio.csv", index=False)
top10.to_csv("absolute_risk_parity_top10.csv", index=False)
print("\nSaved: absolute_risk_parity_portfolio.csv, absolute_risk_parity_top10.csv")

# constraint checks
viol_sectors = sector_exposure[sector_exposure > SECTOR_CAP]
if len(viol_sectors)>0:
    print("\nSectors violating cap:")
    display(viol_sectors.to_frame("Exposure"))
else:
    print("\nNo sector cap violations.")

viol_weights = portfolio[portfolio['Weight'] > MAX_STOCK_WEIGHT + 1e-8]
if len(viol_weights)>0:
    print("\nStocks exceeding max weight:")
    display(viol_weights)
else:
    print("\nNo stocks exceed max weight.")

if not (BETA_LOW - 1e-8 <= portfolio_beta <= BETA_HIGH + 1e-8):
    print(f"\nWarning: portfolio beta {portfolio_beta:.4f} outside band [{BETA_LOW},{BETA_HIGH}]")
else:
    print("\nPortfolio beta within band.")

print("\nDone.")


Fetched sector_map size from Wikipedia: 0
After yfinance fallback, mapping size: 545
Universe size after NIFTY500 strictness: 545
Using 26 tickers for optimization (after data sufficiency filter).
Stage 1: running risk-parity warm-start.
Stage 2: projecting to constraints (sector caps, liquidity, beta band, turnover)
Stage2 projection did NOT converge cleanly: Positive directional derivative for linesearch
Stage2 retry failed: Positive directional derivative for linesearch

--- Absolute-Risk (Risk-Parity) Portfolio Summary ---
Elapsed time: 105.7s
Positions: 26
Total weight: 0.9999999999999998
Top  26 :


,Stock,Weight,Sector
0,BRITANNIA,0.0385,Consumer Defensive
1,BBOX,0.0385,Technology
2,BPCL,0.0385,Energy
3,ABAN,0.0385,Energy
4,BEL,0.0385,Industrials
5,BHEL,0.0385,Industrials
6,ADANIENT,0.0385,Energy
7,GRAPHITE,0.0385,Industrials
8,FEDERALBNK,0.0385,Financial Services
9,CANFINHOME,0.0385,Financial Services



Sector exposures:


,Weight
Sector,
Basic Materials,0.2308
Financial Services,0.1923
Consumer Cyclical,0.1923
Energy,0.1154
Industrials,0.1154
Utilities,0.0769
Consumer Defensive,0.0385
Technology,0.0385



Expected annual return (approx): 0.2951 (29.51%)
Annual volatility (approx): 0.1966 (19.66%)
Portfolio beta: 0.3867
Tracking error vs benchmark (empirical): 0.3026 (30.26%)
Per-rebalance turnover (L1): 0.0000 (cap: 0.2083)

Saved: absolute_risk_parity_portfolio.csv, absolute_risk_parity_top10.csv

No sector cap violations.

No stocks exceed max weight.


Done.
